In [34]:
import re

with open("data/fra.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text.split("\n")
english = []
french = []

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())


for lines in text:
    line = []
    lines = lines.split("\t")
    if len(lines) >= 2:
        english.append(tokenize(lines[0]))
        french.append(["<SOS>"] + tokenize(lines[1]) + ["<EOS>"])

print(english[:10])
print(french[:10])

[['go', '.'], ['go', '.'], ['go', '.'], ['go', '.'], ['hi', '.'], ['hi', '.'], ['run', '!'], ['run', '!'], ['run', '!'], ['run', '!']]
[['<SOS>', 'va', '!', '<EOS>'], ['<SOS>', 'marche', '.', '<EOS>'], ['<SOS>', 'en', 'route', '!', '<EOS>'], ['<SOS>', 'bouge', '!', '<EOS>'], ['<SOS>', 'salut', '!', '<EOS>'], ['<SOS>', 'salut', '.', '<EOS>'], ['<SOS>', 'cours', '!', '<EOS>'], ['<SOS>', 'courez', '!', '<EOS>'], ['<SOS>', 'prenez', 'vos', 'jambes', 'à', 'vos', 'cous', '!', '<EOS>'], ['<SOS>', 'file', '!', '<EOS>']]


In [35]:
import torch

special_tokens = [
    "<PAD>",
    "<SOS>",   # Start of sentence
    "<EOS>",   # End of sentence
    "<UNK>"    # Unknown word
]

Evocab = special_tokens + sorted(set(word for sentence in english for word in sentence))
Fvocab = special_tokens + sorted(set(word for sentence in french for word in sentence))

eng_word2idx = {word: i for i, word in enumerate(Evocab)}
fra_word2idx = {word: i for i, word in enumerate(Fvocab)}

eng_idx2word = {i: word for word, i in eng_word2idx.items()}
fra_idx2word = {i: word for word, i in fra_word2idx.items()}

len(Evocab), len(Fvocab)

(16288, 27245)

In [36]:
pairs = list(zip(english, french))

indexed_pairs = []

for eng, fra in pairs:
    eng_ids = [eng_word2idx.get(word, eng_word2idx["<UNK>"]) for word in eng]
    fra_ids = [fra_word2idx.get(word, fra_word2idx["<UNK>"]) for word in fra]

    indexed_pairs.append((eng_ids, fra_ids))

In [37]:
from torch.nn.utils.rnn import pad_sequence

src_batch = [
    torch.tensor(src, dtype=torch.long)
    for src, _ in indexed_pairs[:32]
]

src_batch = pad_sequence(
    src_batch,
    batch_first=True,
    padding_value=eng_word2idx["<PAD>"]
)

In [38]:
def collate_fn(batch):
    src, tgt = zip(*batch)

    src_lengths = torch.tensor([len(s) for s in src])
    tgt_lengths = torch.tensor([len(t) for t in tgt])

    src = pad_sequence(
        src,
        batch_first=True,
        padding_value=eng_word2idx["<PAD>"]
    )

    tgt = pad_sequence(
        tgt,
        batch_first=True,
        padding_value=fra_word2idx["<PAD>"]
    )

    return src, tgt, src_lengths, tgt_lengths

In [39]:
from torch.utils.data import Dataset, DataLoader,random_split

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]

        src = torch.tensor(src, dtype=torch.long)
        tgt = torch.tensor(tgt, dtype=torch.long)

        return src, tgt

dataset = TranslationDataset(indexed_pairs)

train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    batch_size=32,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available()
)
test_dataloader = DataLoader(
    test_dataset,
    shuffle=True,
    batch_size=32,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available()
)


In [40]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch import nn

class Encoder(nn.Module):
    def __init__(self,embedding_dim,hidden_dim = 128,num_layers =1,dropout = 0.2) -> None:
        super().__init__()
        self.EngEmbedding = nn.Embedding(len(Evocab), embedding_dim,padding_idx=eng_word2idx["<PAD>"])
        self.encoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

    def forward(self,input_ids,lengths):
        embeddings = self.EngEmbedding(input_ids)
        
        packed = pack_padded_sequence(
            embeddings,
            lengths.tolist(),
            batch_first=True,
            enforce_sorted=False
        )
        packed_outputs, (hidden, cell) = self.encoder(packed)
        outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True
        )

        return outputs, hidden, cell

class Decoder(nn.Module):
    def __init__(self,embedding_dim,hidden_dim = 128,num_layers =1,dropout = 0.2) -> None:
        super().__init__()
        self.FraEmbedding = nn.Embedding(len(Fvocab), embedding_dim,padding_idx=fra_word2idx["<PAD>"])
        self.fc = nn.Linear(hidden_dim, len(Fvocab))

        self.decoder =  nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

    def forward(self, decoder_input, hidden, cell):        

        embeddings = self.FraEmbedding(decoder_input)
        outputs, (hidden, cell) = self.decoder(
            embeddings,
            (hidden, cell)
        )
        logits = self.fc(outputs)
        return logits, hidden, cell



class Eng2Fra_Seq2Seq(torch.nn.Module):

    def __init__(self,embedding_dim,hidden_dim = 128,num_layers =1,dropout = 0.2):
        super().__init__()

        self.encoder = Encoder(embedding_dim,hidden_dim,num_layers,dropout)
        self.decoder = Decoder(embedding_dim,hidden_dim,num_layers,dropout)

    def forward(self, src, src_lengths, tgt_input):

        _, hidden, cell = self.encoder(src, src_lengths)

        logits, hidden, cell = self.decoder(tgt_input,hidden,cell)

        return logits
        
        

In [41]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = Eng2Fra_Seq2Seq(
    embedding_dim=128,
    hidden_dim=128
).to(device)

print(model)

Eng2Fra_Seq2Seq(
  (encoder): Encoder(
    (EngEmbedding): Embedding(16288, 128, padding_idx=0)
    (encoder): LSTM(128, 128, batch_first=True)
  )
  (decoder): Decoder(
    (FraEmbedding): Embedding(27245, 128, padding_idx=0)
    (fc): Linear(in_features=128, out_features=27245, bias=True)
    (decoder): LSTM(128, 128, batch_first=True)
  )
)


In [42]:
criterion = nn.CrossEntropyLoss(
    ignore_index=fra_word2idx["<PAD>"]
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for src, tgt, src_lengths, tgt_lengths in train_dataloader:

        src = src.to(device)
        tgt = tgt.to(device)
        src_lengths = src_lengths.cpu()
        
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad()

        logits = model(src,src_lengths, tgt_input)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=-1)

        mask = tgt_output != fra_word2idx["<PAD>"]

        correct += (
            (predictions == tgt_output) & mask
        ).sum().item()

        total += mask.sum().item()

    train_loss = total_loss / len(train_dataloader)
    train_accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"| Loss: {train_loss:.4f} "
        f"| Accuracy: {train_accuracy:.4f}"
    )

Epoch 1/5 | Loss: 2.9107 | Accuracy: 0.5092
Epoch 2/5 | Loss: 1.9643 | Accuracy: 0.6133
Epoch 3/5 | Loss: 1.6290 | Accuracy: 0.6571
Epoch 4/5 | Loss: 1.4244 | Accuracy: 0.6853
Epoch 5/5 | Loss: 1.2805 | Accuracy: 0.7067


In [47]:
length_buckets = [
    (1, 5),
    (6, 10),
    (11, 15),
    (16, 20),
    (21, 25),
    (26, 30),
    (31, 40),
    (41, 1000)
]

from collections import defaultdict

model.eval()

correct_by_bucket = defaultdict(int)
total_by_bucket = defaultdict(int)

with torch.no_grad():

    for src, tgt, src_lengths, tgt_lengths in test_dataloader:

        src = src.to(device)
        tgt = tgt.to(device)
        src_lengths = src_lengths.cpu()

        # Teacher forcing
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        logits = model(
            src,
            src_lengths,
            tgt_input
        )

        predictions = logits.argmax(dim=-1)

        # Ignore PAD tokens
        mask = tgt_output != fra_word2idx["<PAD>"]

        for i in range(src.size(0)):

            src_len = src_lengths[i].item()

            # Find bucket
            for low, high in length_buckets:

                if low <= src_len <= high:

                    example_mask = mask[i]

                    correct = (
                        (predictions[i] == tgt_output[i])
                        & example_mask
                    ).sum().item()

                    total = example_mask.sum().item()

                    correct_by_bucket[(low, high)] += correct
                    total_by_bucket[(low, high)] += total

                    break

for low, high in length_buckets:

    correct = correct_by_bucket[(low, high)]
    total = total_by_bucket[(low, high)]

    if total > 0:
        accuracy = correct / total

        print(
            f"{low:2d}-{high:2d} tokens: "
            f"{accuracy:.4f} "
            f"({total} tokens)"
        )

 1- 5 tokens: 0.7264 (28726 tokens)
 6-10 tokens: 0.7011 (148815 tokens)
11-15 tokens: 0.6173 (43499 tokens)
16-20 tokens: 0.5118 (6920 tokens)
21-25 tokens: 0.4436 (1889 tokens)
26-30 tokens: 0.4130 (322 tokens)
31-40 tokens: 0.2794 (204 tokens)
41-1000 tokens: 0.4348 (46 tokens)
